# 18.1 模型部署与服务 (Model Serving)

> 🕐 预估学习时间：40分钟

将训练好的模型推向生产环境，实现高性能推理服务。本节涵盖主流的模型部署方案和关键技术。

本节涵盖：
- 推理引擎对比（vLLM、TGI、Triton）
- 模型 Serving 架构设计
- 动态批处理与调度
- API 网关与负载均衡
- 成本优化策略

## 1. 主流推理引擎对比

| 引擎 | 核心优势 | 适用场景 |
|------|----------|----------|
| **vLLM** | PagedAttention、Continuous Batching | 高吞吐 LLM 推理 |
| **TGI (Text Generation Inference)** | HuggingFace 原生、水印、语法约束 | HF 生态用户 |
| **Triton Inference Server** | 多框架、多模型、动态批处理 | 企业级多模型服务 |
| **TensorRT-LLM** | NVIDIA 深度优化、FP8 | 追求极致延迟 |
| **SGLang** | RadixAttention、结构化生成 | 结构化输出场景 |

**选型建议**：
- 通用场景：vLLM（社区活跃、迭代快）
- HuggingFace 生态：TGI
- 多模型管理 + 传统 ML：Triton
- 极致性能 + NVIDIA GPU：TensorRT-LLM

In [ ]:
import torch
import torch.nn as nn
import time
from collections import deque

torch.manual_seed(42)

class SimpleServingEngine:
    def __init__(self, model, max_batch_size=32, max_waiting_time=0.05):
        self.model = model
        self.max_batch_size = max_batch_size
        self.max_waiting_time = max_waiting_time
        self.request_queue = deque()
        self.stats = {'total_requests': 0, 'avg_latency': 0, 'avg_batch_size': 0}

    def add_request(self, input_tensor):
        self.request_queue.append((time.time(), input_tensor))

    def process_batch(self):
        if not self.request_queue:
            return None, 0

        batch = []
        current_time = time.time()
        while self.request_queue and len(batch) < self.max_batch_size:
            arrival_time, request = self.request_queue[0]
            if current_time - arrival_time >= self.max_waiting_time or len(batch) > 0:
                self.request_queue.popleft()
                batch.append((arrival_time, request))
            else:
                break

        if not batch:
            return None, 0

        inputs = torch.stack([r[1] for _, r in batch])
        with torch.no_grad():
            outputs = self.model(inputs)

        latencies = [current_time - arr for arr, _ in batch]
        return outputs, latencies

model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64))
engine = SimpleServingEngine(model, max_batch_size=16, max_waiting_time=0.05)

for _ in range(50):
    engine.add_request(torch.randn(64))

print('=== Dynamic Batching Server ===')
total_latency = 0
n_batches = 0
while True:
    outputs, latencies = engine.process_batch()
    if outputs is None:
        break
    batch_size = len(latencies)
    avg_lat = sum(latencies) / batch_size * 1000
    total_latency += avg_lat * batch_size
    n_batches += 1
    if n_batches <= 5:
        print(f'  Batch {n_batches}: {batch_size} requests, avg latency={avg_lat:.1f}ms')

print(f'\nTotal: {n_batches} batches, avg batch size: {50/n_batches:.1f}')
print(f'\nKey: Dynamic batching waits to form larger batches, improving throughput.')
print(f'max_waiting_time balances latency vs throughput.')

## 2. 模型 Serving 架构

**生产级架构**：

```
用户请求 → API Gateway(Nginx/Kong) → 负载均衡 → 推理服务集群(vLLM) → 返回
                                          ↓
                                    模型注册中心
                                    (模型版本管理)
                                          ↓
                                    监控 & 日志
                                  (Prometheus + Grafana)
```

**关键组件**：
- **API Gateway**：认证、限流、路由
- **Load Balancer**：轮询/最少连接分发请求
- **Model Registry**：模型版本管理与回滚
- **Health Check**：定期健康检查，自动摘除故障实例

In [ ]:
import random
from dataclasses import dataclass

@dataclass
class ModelInstance:
    id: str
    model_version: str
    status: str
    load: float

class LoadBalancer:
    def __init__(self, strategy='least_connections'):
        self.strategy = strategy
        self.instances = []

    def register(self, instance):
        self.instances.append(instance)

    def select_instance(self):
        healthy = [i for i in self.instances if i.status == 'healthy']
        if not healthy:
            return None
        if self.strategy == 'least_connections':
            return min(healthy, key=lambda i: i.load)
        elif self.strategy == 'round_robin':
            return healthy[hash(str(time.time())) % len(healthy)]
        return random.choice(healthy)

class ModelRegistry:
    def __init__(self):
        self.models = {}

    def register(self, name, version, path, metadata=None):
        if name not in self.models:
            self.models[name] = []
        self.models[name].append({
            'version': version, 'path': path,
            'metadata': metadata or {}, 'timestamp': time.time()
        })

    def get_latest(self, name):
        versions = self.models.get(name, [])
        return max(versions, key=lambda v: v['version']) if versions else None

    def rollback(self, name, version):
        for v in self.models.get(name, []):
            if v['version'] == version:
                return v
        return None

balancer = LoadBalancer()
registry = ModelRegistry()

for i in range(3):
    balancer.register(ModelInstance(f'instance-{i}', 'v1.0', 'healthy', random.random()))
balancer.register(ModelInstance('instance-3', 'v1.0', 'unhealthy', 0))

registry.register('llama-chat', '1.0', '/models/llama-v1.0', {'perf': 'baseline'})
registry.register('llama-chat', '1.1', '/models/llama-v1.1', {'perf': '+15%'})

print('=== Model Serving Architecture ===')
print(f'\nLoad Balancer ({balancer.strategy}):')
for inst in balancer.instances:
    print(f'  {inst.id}: status={inst.status}, load={inst.load:.2f}')

selected = balancer.select_instance()
print(f'\nSelected instance: {selected.id if selected else "none"}')

print(f'\nModel Registry:')
latest = registry.get_latest('llama-chat')
print(f'  Latest: version={latest["version"]}, path={latest["path"]}')
rolled = registry.rollback('llama-chat', '1.0')
print(f'  Rollback to 1.0: path={rolled["path"]}')

print(f'\nKey: Load balancer routes to healthy instances with lowest load.')
print(f'Model registry enables version management and fast rollback.')

## 3. 成本优化

**推理成本构成**：
- GPU 实例成本（主要）
- 网络带宽成本
- 存储成本

**优化策略**：

1. **量化**：INT8/INT4 量化降低显存和计算
2. **KV Cache 优化**：PagedAttention + KV 量化
3. **投机解码**：小模型 draft + 大模型验证
4. **模型蒸馏**：用小模型处理简单请求
5. **弹性伸缩**：根据负载自动扩缩实例
6. **缓存命中**：常见问题直接返回缓存结果

**成本估算模型**：

In [ ]:
def estimate_serving_cost(
    model_size_b=70,
    requests_per_second=100,
    avg_tokens_per_request=500,
    gpu_cost_per_hour=3.0,
    tokens_per_second_per_gpu=2000
):
    total_tokens_per_sec = requests_per_second * avg_tokens_per_request
    gpus_needed = total_tokens_per_sec / tokens_per_second_per_gpu
    daily_cost = gpus_needed * gpu_cost_per_hour * 24

    print('=== Inference Cost Estimation ===')
    print(f'Model: {model_size_b}B parameters')
    print(f'Load: {requests_per_second} req/s × {avg_tokens_per_request} tokens = {total_tokens_per_sec:,} tokens/s')
    print(f'GPUs needed: {gpus_needed:.1f}')
    print(f'Daily cost: ${daily_cost:,.0f}')
    print(f'Monthly cost: ${daily_cost * 30:,.0f}')

    print(f'\n--- Optimization Scenarios ---')
    for method, improvement in [
        ('INT8 Quantization', 0.5),
        ('Speculative Decoding', 0.6),
        ('INT4 + Spec Decode', 0.3),
    ]:
        opt_gpus = gpus_needed * improvement
        opt_daily = opt_gpus * gpu_cost_per_hour * 24
        saving = daily_cost - opt_daily
        print(f'  {method}: {opt_gpus:.1f} GPUs, ${opt_daily:,.0f}/day (save ${saving:,.0f}/day)')

estimate_serving_cost()

print(f'\nKey: Quantization + Speculative Decoding can reduce costs by 50-70%.')
print(f'Semantic caching can further reduce costs for repeated queries.')

## 📝 课后思考题

1. Continuous Batching 如何比 Static Batching 提高 GPU 利用率？
2. 在多模型服务的场景下，如何设计公平的 GPU 资源调度策略？
3. 模型 A/B 测试时，如何确保流量分配的一致性（同一用户始终路由到同一模型）？